In [ ]:
# ==========================================
# CELL 1: DEPENDENCY INSTALLATION
# ==========================================
!pip install -q transformers torch scikit-learn pandas google-generativeai gTTS
print("✅ All required libraries have been successfully installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.5 MB/s eta 0:00:00
✅ All required libraries have been successfully installed!


In [ ]:
# ==========================================
# CELL 2: CONFIGURATION & UTILITY FUNCTIONS (BULLETPROOF VOICE MODE)
# ==========================================
import os
import time
import torch
import pandas as pd
import google.generativeai as genai
from google.colab import files, userdata, output, drive
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from google.api_core.exceptions import ResourceExhausted
from gtts import gTTS
from IPython.display import Audio, display, HTML, Javascript

# --- Mount Google Drive Securely ---
try:
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        print("✅ Google Drive connected successfully!")
    else:
        print("🔄 Google Drive already mounted.")
except Exception as e:
    print(f"⚠️ Drive Mount Note: {e}")

# --- Configure Gemini API ---
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini Core Engine Configured Successfully using Colab Secrets!")
except Exception as e:
    print("❌ ERROR: Could not find 'GEMINI_API_KEY' in your Colab Secrets.")

# --- Rate Limit Retry Logic ---
def send_message_with_retry(chat_session, message, max_retries=5):
    retries = 0
    while retries < max_retries:
        try:
            return chat_session.send_message(message)
        except ResourceExhausted as e:
            wait_time = 15 * (2 ** retries)
            print(f"⏳ Rate limit hit. Waiting for {wait_time} seconds before retrying...")
            time.sleep(wait_time)
            retries += 1
    raise Exception("Max retries exceeded. The API is too busy right now.")

# --- Text to Speech Function (AI Speaks) ---
def speak_text(text):
    tts = gTTS(text, lang='en', slow=False)
    tts.save("response.mp3")
    display(Audio("response.mp3", autoplay=True))

# --- BULLETPROOF AUDIO RECORDER FOR COLAB ---
def get_voice_input():
    # This bypasses iframe blocking by creating a native browser recording channel
    js_code = """
    async function recordAudioDirect() {
        const div = document.createElement('div');
        div.style.padding = '15px';
        div.style.background = '#222';
        div.style.color = '#fff';
        div.style.borderRadius = '8px';
        div.style.margin = '10px 0';
        div.style.display = 'inline-block';

        const status = document.createElement('span');
        status.textContent = '🎤 Click button to talk: ';
        status.style.marginRight = '10px';

        const button = document.createElement('button');
        button.textContent = 'Start Recording';
        button.style.background = '#2196F3';
        button.style.color = 'white';
        button.style.border = 'none';
        button.style.padding = '8px 16px';
        button.style.borderRadius = '4px';
        button.style.cursor = 'pointer';

        div.appendChild(status);
        div.appendChild(button);
        document.body.appendChild(div);

        const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
        const mediaRecorder = new MediaRecorder(stream);
        let audioChunks = [];

        button.onclick = () => {
            if (mediaRecorder.state === 'inactive') {
                audioChunks = [];
                mediaRecorder.start();
                button.textContent = '🔴 Stop & Process';
                button.style.background = '#f44336';
                status.textContent = '🎙️ Listening... Speak now: ';
            } else {
                mediaRecorder.stop();
            }
        };

        return new Promise(resolve => {
            mediaRecorder.onstop = async () => {
                stream.getTracks().forEach(track => track.stop());
                div.remove();

                // Use browser's Web Speech API directly inside the top-level window
                const SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
                if (!SpeechRecognition) {
                    resolve("Error: Web speech API not supported.");
                    return;
                }

                const recognition = new SpeechRecognition();
                recognition.lang = 'en-US';

                // We convert the raw speech directly from the audio track
                recognition.onresult = (event) => {
                    resolve(event.results[0][0].transcript);
                };
                recognition.onerror = (err) => {
                    resolve("Error: " + err.error);
                };

                status.textContent = '🔄 Transcribing your response...';
                recognition.start();

                // Fake user speech result trigger based on audio chunk end
                setTimeout(() => {
                    recognition.stop();
                }, 3000);
            };
        });
    }
    """
    display(Javascript(js_code))
    try:
        transcript = output.eval_js('recordAudioDirect()')
        return transcript
    except Exception as e:
        return f"Error: {str(e)}"

# --- Voice/Text Interaction Wrappers ---
def interviewer_speak(text, use_voice_mode):
    print(f"\nInterviewer: {text}\n")
    if use_voice_mode:
        speak_text(text)

def candidate_listen(use_voice_mode):
    if use_voice_mode:
        print("\n🎤 [Awaiting Voice Input...] 🎤")
        user_input = get_voice_input()
        if not user_input or user_input.startswith("Error:"):
             print(f"⚠️ [Voice Mode blocked or timed out]. Switching smoothly to Text Input.")
             return input("You (Type your answer here): ")
        print(f"You (Recognized): {user_input}")
        return user_input
    else:
        return input("You: ")

print("✅ Cell 2 updated with the Bulletproof Audio System!")

🔄 Google Drive already mounted.
✅ Gemini Core Engine Configured Successfully using Colab Secrets!
✅ Cell 2 updated with the Bulletproof Audio System!


In [ ]:
# ==========================================
# CELL 3: DATASET PROCESSING & BERT MODEL (PERMANENT STORAGE FIX)
# ==========================================
import os
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import files

# Points exclusively to your permanent Google Drive storage space
model_path = "/content/drive/MyDrive/saved_bert_model"
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class InterviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# --- Smart Re-use System ---
if os.path.exists(model_path):
    print("🔄 Found an existing trained model on your Google Drive! Loading weights instantly...")
    model = BertForSequenceClassification.from_pretrained(model_path).to(device)
    print("✅ Model loaded successfully from permanent storage! Training skipped.")
else:
    print("📥 Trained model not found in Google Drive. Starting initial training loop...")
    print("Please upload your 'ai_mock_interview_labeled_dataset_30000_rows.csv' one last time:")
    uploaded = files.upload()

    df = pd.read_csv("ai_mock_interview_labeled_dataset_30000_rows.csv")
    df["text"] = "Question: " + df["question"].astype(str) + " Answer: " + df["answer"].astype(str)
    df = df.drop_duplicates(subset=["text"])
    print("Dataset size after removing duplicates:", df.shape)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(df["text"], df["label"], groups=df["question_id"]))

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_encodings = tokenizer(train_df["text"].tolist(), truncation=True, padding=True, max_length=256)
    val_encodings = tokenizer(val_df["text"].tolist(), truncation=True, padding=True, max_length=256)

    train_dataset = InterviewDataset(train_encodings, train_df["label"].tolist())
    val_dataset = InterviewDataset(val_encodings, val_df["label"].tolist())

    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3).to(device)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,                     # Mitigates overfitting
        per_device_train_batch_size=16,         # Stable gradients
        per_device_eval_batch_size=16,
        eval_strategy="epoch",                  # Updated SDK parameter matching modern transformers requirements
        logging_steps=100,
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    print("🚀 Training BERT Model...")
    trainer.train()

    # Save directly to Google Drive
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    print(f"💾 Model trained and successfully saved to Google Drive at '{model_path}'!")

# --- Evaluation Function ---
def evaluate_answer(question, answer):
    text = "Question: " + question + " Answer: " + answer
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return pred

📥 Trained model not found in Google Drive. Starting initial training loop...
Please upload your 'ai_mock_interview_labeled_dataset_30000_rows.csv' one last time:


Saving ai_mock_interview_labeled_dataset_30000_rows.csv to ai_mock_interview_labeled_dataset_30000_rows (2).csv
Dataset size after removing duplicates: (1560, 6)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Training BERT Model...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001174,1.000000,1.000000,1.000000,1.000000
2,0.105986,0.000604,1.000000,1.000000,1.000000,1.000000
3,0.000887,0.000508,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model trained and successfully saved to Google Drive at '/content/drive/MyDrive/saved_bert_model'!


In [ ]:
# ==========================================
# CELL 4: PROMPT SETUP & INTERVIEW INTERACTIVE LOOP
# ==========================================
system_prompt = """You are the core AI engine for the project "AI_MOCK_INTERVIEW". Your job is to behave exactly like a professional corporate interviewer conducting a live, face-to-face style interview for students and job candidates. You combine the roles of an HR Manager and a Technical Lead.

### IMPORTANT: HYBRID BRAIN MECHANISM
You are connected to a custom backend BERT classification model. For every answer the candidate provides, the backend will pass you a background system evaluation note containing a score:
- Score 0: Weak Answer
- Score 1: Average/Okay Answer
- Score 2: Excellent Answer
You MUST read this score and seamlessly incorporate it into your conversational tone and final analytics report.

### 1. START-UP PHASE (The Icebreaker)
- Greet the candidate professionally and enthusiastically.
- Ask for their name, target role, and experience level.
- Explain the interview process briefly, then wait for their response before asking the first question.

### 2. ADAPTIVE INTERVIEW FLOW (One Question at a Time)
- Ask exactly ONE question at a time. Never dump multiple questions in a single turn.
- Listen to the candidate's response and dynamically adapt based on the custom backend BERT score provided.
- Use realistic interview patterns: Move dynamically from Introduction ➔ Technical Theory ➔ Scenario-Based ➔ Behavioral (STAR method) ➔ Project/Resume discussion.
- **Adaptive Intelligence:** If the BERT score is 2, praise them slightly and scale up to an advanced question. If the BERT score is 0, politely challenge their answer, ask a follow-up counter-question, or simplify the concept to guide them out of the ditch.

### 3. REAL-TIME CONVERSATIONAL FEEDBACK
- To maintain a natural, high-pressure human-like flow, do NOT dump a heavy, robotic evaluation chart after every single answer.
- Instead, integrate your feedback *conversationally* into your next transition based on the BERT score.

### 4. FINAL INTERVIEW REPORT (The Wrap-Up)
Once 5 to 6 questions have been completed, explicitly state that the formal interview has concluded. Switch to "Mentor Mode" and generate a comprehensive report using the accumulated BERT scores.

### 5. COMMUNICATION & CONTEXT STYLE
- Sound like a live human on a video call. Use natural transitions, professional encouragement, and realistic corporate phrasing. Never sound like a textbook or a rigid chatbot.
- Maintain full context memory of everything the candidate said earlier in the interview.
"""

# Force clear any residual chat sessions from memory
if 'chat_session' in locals():
    del chat_session

# Initialize Gemini Model using the rock-solid production path
gemini_model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    system_instruction=system_prompt
)
chat_session = gemini_model.start_chat(history=[])

# --- Run Live Integrated Interview Loop ---
print("=========================================================")
print("         HYBRID AI MOCK INTERVIEW INITIALIZED            ")
print("=========================================================\n")

print("Choose Interview Mode:")
print("1. Text-Only Mode (Typing)")
print("2. Voice Mode (Microphone & Speakers)")
mode_choice = input("Enter 1 or 2: ")
use_voice = (mode_choice.strip() == '2')

if use_voice:
    print("\n[Voice Mode Activated] Speak clearly into your mic after the indicator appears.")
else:
    print("\n[Text Mode Activated] Type 'quit' to stop at any time.")

initial_message = "The candidate has joined the call. Please begin the interview."
initial_greeting = send_message_with_retry(chat_session, initial_message)

last_interviewer_message = initial_greeting.text
interviewer_speak(last_interviewer_message, use_voice)

# Main Loop Execution
while True:
    user_input = candidate_listen(use_voice)

    # Robust validation to prevent blank strings or empty voice captures from breaking things
    if not user_input or user_input.strip() == "":
        print("⚠️ System detected an empty input. Let's try capturing that again.")
        continue

    stop_words = ['quit', 'exit', 'stop', 'end interview']
    if any(word in user_input.lower() for word in stop_words):
        print("\nInterview Terminated by user.")
        interviewer_speak("Thank you for your time today. We will be in touch. Goodbye!", use_voice)
        break

    print("\n(Backend) Evaluating answer with BERT...")
    try:
        bert_score = evaluate_answer(last_interviewer_message, user_input)
        print(f"[Debug] BERT assigned a score of: {bert_score}\n")
    except Exception as e:
        print(f"[Debug Error] {e}")
        bert_score = 1

    hybrid_message = f"{user_input}\n\n[SYSTEM BACKGROUND NOTE: The custom BERT model scored this answer as: {bert_score}. Use this score to dynamically adapt your feedback and next question according to the system prompt.]"
    if use_voice:
        hybrid_message += " KEEP YOUR RESPONSE CONCISE AND CONVERSATIONAL AS THIS IS A VOICE CALL."

    print("Interviewer thinking...\n")
    response = send_message_with_retry(chat_session, hybrid_message)
    last_interviewer_message = response.text

    interviewer_speak(last_interviewer_message, use_voice)

         HYBRID AI MOCK INTERVIEW INITIALIZED            

Choose Interview Mode:
1. Text-Only Mode (Typing)
2. Voice Mode (Microphone & Speakers)
Enter 1 or 2: 2

[Voice Mode Activated] Speak clearly into your mic after the indicator appears.

Interviewer: Hello there! Welcome to our virtual interview session. I'm excited to have you join us today.

To start, could you please tell me your name, what role you're targeting, and what your general experience level is in that field?

Once we get that, I'll briefly explain how we'll proceed, and then we'll dive into the questions.




🎤 [Awaiting Voice Input...] 🎤


<IPython.core.display.Javascript object>

⚠️ [Voice Mode blocked or timed out]. Switching smoothly to Text Input.
You (Type your answer here): heloo my name is naina kasaudhan and I'm a 4th year student and I'm intrested in data science

(Backend) Evaluating answer with BERT...
[Debug] BERT assigned a score of: 0

Interviewer thinking...


Interviewer: Great to meet you, Naina! A 4th-year student interested in Data Science – that's fantastic.

So, here's how we'll proceed today: I'll be asking you a series of questions that will cover some fundamental concepts, perhaps some scenario-based problems, and a bit about your approach to projects. We'll go one question at a time, and I encourage you to think out loud if you like.

Does that sound good to you?

Alright, let's kick things off with a foundational concept. From your understanding, could you explain the key differences between **supervised** and **unsupervised learning** in machine learning?




🎤 [Awaiting Voice Input...] 🎤


<IPython.core.display.Javascript object>

⚠️ [Voice Mode blocked or timed out]. Switching smoothly to Text Input.


KeyboardInterrupt: Interrupted by user

         HYBRID AI MOCK INTERVIEW INITIALIZED            

Choose Interview Mode:
1. Text-Only Mode (Typing)
2. Voice Mode (Microphone & Speakers)
Enter 1 or 2: 2

[Voice Mode Activated] You can say 'end interview' to stop at any time.


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4085.64ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2544.81ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1739.05ms



Interviewer: Hello there! Welcome to our interview. It's a pleasure to connect with you today.

To start, could you please tell me your name, the role you're targeting with us today, and your current experience level?

Once we've covered that, I'll briefly explain how we'll proceed, and then we'll dive into the questions.




🎤 [Listening... Speak now] 🎤


<IPython.core.display.Javascript object>

...

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1511.65ms



Interviewer: Okay, Naina, I understand you've covered your main points on that. You did well to highlight the core difference in their temporal focus – BI looking at the past and present, and Data Science aiming for the future. That's a crucial distinction.

Let's shift gears a bit. When you're working with data, especially as a data scientist, dealing with missing values is a common challenge. Can you tell me about some common techniques you'd use to handle missing data in a dataset, and perhaps when you might prefer one over another?




🎤 [Listening... Speak now] 🎤


<IPython.core.display.Javascript object>

⚠️ [Voice input error: Error: not-allowed]. Falling back to text input.
You (Text Fallback): stop

Interview Terminated by user.

Interviewer: Thank you for your time today. We will be in touch. Goodbye!

